In [1]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pickle

save_file_name = 'morpho3d2.pkl'

In [2]:
def cut_sphere(normals):
    nF = len(normals)
    ds = np.sqrt(np.sum(normals**2,axis=1))
    ang0 = np.arccos(ds)
    if nF>1:
        cs = normals@normals.T/np.outer(ds,ds)
        angs = np.arccos(np.minimum(cs,1))
        tanb = (ds[:,None]/ds[None,:]-cs)/np.sin(angs+np.eye(nF))
        cont = tanb/np.tan(ang0)
        n0 = normals/ds[:,None]
        n1 = np.roll(n0,-1,axis=0)
        n1 = n1-np.sum(n0*n1,axis=1,keepdims=True)*n0
        n1 = n1/np.linalg.norm(n1,axis=1,keepdims=True)
        n2 = np.cross(n0,n1)
        x = normals@n1.T
        y = normals@n2.T
        angles = np.mod(np.arctan2(y,x),2*np.pi)
    else:
        cont = np.zeros((1,1))
        angles = np.zeros((1,1))
    
    return cont.T, angles.T, ang0

In [3]:
def cut_circle(cont,angles,ang0):
    nL = len(cont)
    if nL>0:
        if np.min(cont)<=-1:
            return 0.0, 0.0
        sg = np.sign(cont)
        dtheta = np.mod(angles[:,None]-angles[None,:]+np.pi,2*np.pi)-np.pi
        ang1 = np.arccos(cont)
        tanb = (cont[None,:]/cont[:,None]-np.cos(dtheta))/(np.sin(np.abs(dtheta))+np.eye(nL))
        b = np.arctan(tanb)+np.pi*(1-sg[:,None])/2
        a1 = (b-ang1[:,None])*(dtheta>0)+ang1[:,None]
        a2 = (b-ang1[:,None])*(dtheta<0)+ang1[:,None]
        c1 = np.min(a1,axis=1)*(sg+1)/2+np.max(a1,axis=1)*(1-sg)/2
        c2 = np.min(a2,axis=1)*(sg+1)/2+np.max(a2,axis=1)*(1-sg)/2
        c12 = np.clip(c1+c2,0,2*np.pi)
        c12n = np.minimum(c1+c2,2*np.pi-c1-c2)
        area1 = np.maximum((np.tan(c1)+np.tan(c2))*sg,0)*cont**2/2*sg
        area2 = np.maximum(np.pi*np.maximum(np.sum((1-sg)/2),1)-np.sum(c12)/2,0)
        area = np.sum(area1)+area2
        area = area*np.sin(ang0)**2
        a = np.arctan2(np.sin(ang0)*np.abs(cont),np.cos(ang0))
        volume1 = (c12n-np.arcsin(np.cos(a)*np.sin(c2))-np.arcsin(np.cos(a)*np.sin(c1)))/3*sg*np.heaviside(c12n,0)
        volume2 = area2*2/3*(1-np.cos(ang0))
        dvolume = np.sum(volume1)+volume2-area*np.cos(ang0)/3
    else:
        area = np.pi*np.sin(ang0)**2
        dvolume = np.pi*2/3*(1-np.cos(ang0))-area*np.cos(ang0)/3
    return area, dvolume

In [4]:
def cell_geometry(x,y,z,u):
    nC = len(x)
    dx = -x[:,None]+x[None,:]
    dy = -y[:,None]+y[None,:]
    dz = -z[:,None]+z[None,:]
    du = u[:,None]**2-u[None,:]**2
    mask = ~np.eye(nC, dtype=bool)
    dx = dx[mask].reshape(nC,nC-1)
    dy = dy[mask].reshape(nC,nC-1)
    dz = dz[mask].reshape(nC,nC-1)
    du = du[mask].reshape(nC,nC-1)
    dq = np.sqrt(dx**2+dy**2+dz**2)
    cs = (du+dq**2)/(2*dq*u[:,None])
    nx = dx*cs/dq
    ny = dy*cs/dq
    nz = dz*cs/dq
    normals = np.stack((nx,ny,nz),axis=2)
    return normals, cs

In [5]:
def adjacency_matrix(x,y,z,u):
    normals,cs = cell_geometry(x,y,z,u)
    nC = len(x)
    A = np.zeros((nC,nC-1))
    volumes = np.zeros(nC)
    for cc in range(nC):
        ff = np.where(np.abs(cs[cc])<1)[0]
        nF = len(ff)
        volumec = np.pi*4/3
        if nF>0:
            normalc = normals[cc].copy()
            normalc = normalc[ff,:]
            cont,angles,ang0 = cut_sphere(normalc)
            for ii in range(nF):
                jj = np.where(cont[ii,:]<1)[0]
                jj = np.setdiff1d(jj,ii)
                conti = cont[ii,jj]
                anglesi = angles[ii,jj]
                area,dvolume = cut_circle(conti,anglesi,ang0[ii])
                volumec -= dvolume
                A[cc,ff[ii]] = area*u[cc]**2
        volumes[cc] = volumec*u[cc]**3
    adjacency = np.zeros((nC,nC))
    for i in range(nC):
        adjacency[i, :i] = A[i, :i]
        adjacency[i, i+1:] = A[i, i:]
    return adjacency, volumes

In [6]:
def anisotropy(x,y,z,a,c):
    dx = x-x[c]
    dy = y-y[c]
    dz = z-z[c]
    dl = np.sqrt(dx**2+dy**2+dz**2)
    dl[c] += 1
    dx = dx/dl
    dy = dy/dl
    dz = dz/dl
    e11 = np.sum((1-dx*dx)*a)
    e22 = np.sum((1-dy*dy)*a)
    e33 = np.sum((1-dz*dz)*a)
    e12 = np.sum(-dx*dy*a)
    e13 = np.sum(-dx*dz*a)
    e23 = np.sum(-dy*dz*a)
    st = np.array([[e11,e12,e13],[e12,e22,e23],[e13,e23,e33]])
    w, v = np.linalg.eig(st)
    first_index = np.argmax(w)
    return v[:, first_index]

In [7]:
def MechanicRule(M,x,y,z,u,volumes,adjacency):
    cellnum = len(x)
    dx = 0.1
    dw = 0.1
    v0 = 20
    t0 = 2
    tns = np.tile((M[1,:]+.1)*t0,(cellnum,1))
    vl = v0*2/3*(1+M[0,:])
    
    adj0 = np.sign(adjacency)
    nxs = np.tile(x,(cellnum,1))
    nys = np.tile(y,(cellnum,1))
    nzs = np.tile(z,(cellnum,1))
    nws = np.tile(u**2,(cellnum,1))
    dnx = nxs-nxs.T
    dny = nys-nys.T
    dnz = nzs-nzs.T
    tn = tns+tns.T
    dq = np.sqrt(dnx**2+dny**2+dnz**2)+np.eye(cellnum)
    mx1 = np.sum(dnx/dq*(tn-dq)*adj0,axis=0)
    my1 = np.sum(dny/dq*(tn-dq)*adj0,axis=0)
    mz1 = np.sum(dnz/dq*(tn-dq)*adj0,axis=0)
    vls = np.tile(vl-volumes,(cellnum,1))
    dnw = (nws-nws.T)/(dq**2)
    mx2 = np.sum(dnx/dq*(vls*(1-dnw)+vls.T*(1+dnw))*adjacency,axis=0)
    my2 = np.sum(dny/dq*(vls*(1-dnw)+vls.T*(1+dnw))*adjacency,axis=0)
    mz2 = np.sum(dnz/dq*(vls*(1-dnw)+vls.T*(1+dnw))*adjacency,axis=0)
    mw1 = np.sum((vls-vls.T)*adjacency/dq,axis=0)*u
    mw2 = (vl-volumes)*(4*np.pi*u*u-np.sum(adjacency,axis=0))

    mx = (mx1/(t0**2)+mx2/(v0**2))*dx
    my = (my1/(t0**2)+my2/(v0**2))*dx
    mz = (mz1/(t0**2)+mz2/(v0**2))*dx
    mu = (mw1+mw2)/(v0**2)*dw
    x += np.sign(mx)*np.minimum(np.abs(mx),dx)
    y += np.sign(my)*np.minimum(np.abs(my),dx)
    z += np.sign(mz)*np.minimum(np.abs(mz),dx)
    u += np.sign(mu)*np.minimum(np.abs(mu),dw)
    
    return x, y, z, u

In [8]:
def AutomataRule(M,adj,W):
    cellnum = len(M[0,:])
    nM = np.matmul(M,adj) #compute the neighbor effects
    A = np.vstack((M[2:,:],nM[2:,:],np.ones((1,cellnum))))
    S = np.matmul(W,A)
    M[2:,:] = 1/(np.exp(-S)+1)
    M[0,:] = M[0,:]+0.01/(1+np.exp((M[4,:]-.5)*10)) # M0 increase for cell growth
    return M

In [9]:
def CellDivision(M,x,y,z,u,volumes,adjacency):
    cellnum = len(x)
    x0,y0,z0 = x,y,z
    for ii in range(cellnum):
        if M[0,ii]>0.9:
            M[0,ii] = 0
            v = anisotropy(x0,y0,z0,adjacency[ii],ii)
            v = v*0.2+np.random.normal(loc=0,scale=0.01,size=3)
            x = np.append(x,x[ii]+v[0])
            y = np.append(y,y[ii]+v[1])
            z = np.append(z,z[ii]+v[2])
            u = np.append(u,u[ii])
            x[ii] -= v[0]
            y[ii] -= v[1]
            z[ii] -= v[2]
            M = np.column_stack((M,M[:,ii]))
            #return M,x,y,z,u
    return M,x,y,z,u

In [10]:
cellnum = 1
molnum = 5
x = np.array([1.0])
y = np.array([1.0])
z = np.array([1.0])
u = np.array([1.0])
M = np.ones((molnum,cellnum))*0.5
W = np.zeros((molnum-2,2*molnum-3))
W[1,3] = 0.02
W[1,4] = 0.10
W[1,6] = -2.50
W[2,1] = 1.00
W[2,5] = 0.25
W[2,6] = -3.00

In [11]:
adjacency,volumes = adjacency_matrix(x,y,z,u)
x_list = [x.copy()]
y_list = [y.copy()]
z_list = [z.copy()]
u_list = [u.copy()]
v_list = [volumes.copy()]
m3_list = [M[3].copy()]
m4_list = [M[4].copy()]
for t in range(5500):
    x,y,z,u = MechanicRule(M,x,y,z,u,volumes,adjacency)
    M,x,y,z,u = CellDivision(M,x,y,z,u,volumes,adjacency)
    adjacency,volumes = adjacency_matrix(x,y,z,u)
    if np.mod(t,10)==0:
        x_list.append(x.copy())
        y_list.append(y.copy())
        z_list.append(z.copy())
        u_list.append(u.copy())
        v_list.append(volumes.copy())
        m3_list.append(M[3].copy())
        m4_list.append(M[4].copy())
        M = AutomataRule(M,adjacency,W)
        print(t,end='\r')

5490

In [12]:
with open(save_file_name,'wb') as pk:
    pickle.dump((x_list,y_list,z_list,u_list,v_list,m3_list,m4_list),pk)